# 面试问题：Contextual Bandit 的 LinUCB 怎样在反馈延迟时探索，并避免未来信息泄漏？

        ## 可直接复述的回答主线

        1. 贪心只用已观察均值，冷启动时容易永久选择第一个动作，得不到其他动作的反馈。
2. LinUCB 为每个 arm 维护 A 和 b，用 theta=A^-1b 估计收益，并加入 alpha*sqrt(x^T A^-1 x) 探索奖励。
3. 反馈延迟时决策只允许应用 due_time<=当前时刻的事件，未到反馈保存在 pending ledger。
4. 反馈必须通过 interaction_id 关联原始 context 和 chosen arm，不能按到达顺序猜测归属。
5. 离线回放应逐样本输出预测均值、置信 bonus、选择、真实收益、pending 数和累计 regret。
6. 生产还需 propensity、反事实评估、奖励窗口、去重、探索预算、约束动作、安全兜底和非平稳遗忘。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是十八次客服帮助方式选择，动作包括知识文章、视频教程和人工回拨。上下文为 bias、紧急度、技术问题标记；离线日志给出三种动作的反事实满意度，只用于教学比较，线上实际只能看到被选动作的延迟反馈。

In [1]:
import math  # 计算累计 regret 和 UCB 置信项。
import numpy as np  # 手写 LinUCB 矩阵更新与求逆。
arms = ["article", "video", "callback"]  # 定义三种可选客服干预。
event_specs = [("bandit-01", "退款规则查询", 0.15, 0, "article", 3), ("bandit-02", "发票抬头查询", 0.20, 0, "article", 2), ("bandit-03", "SDK鉴权失败", 0.65, 1, "video", 4), ("bandit-04", "账户被盗紧急处理", 0.95, 0, "callback", 2), ("bandit-05", "批量接口调试", 0.55, 1, "video", 3), ("bandit-06", "物流时效查询", 0.10, 0, "article", 2), ("bandit-07", "支付接口报错", 0.80, 1, "video", 3), ("bandit-08", "大额扣款申诉", 0.90, 0, "callback", 4), ("bandit-09", "会员续费规则", 0.25, 0, "article", 2), ("bandit-10", "Webhook签名验证", 0.60, 1, "video", 3), ("bandit-11", "账号冻结急需恢复", 0.88, 0, "callback", 2), ("bandit-12", "退货地址查询", 0.18, 0, "article", 3), ("bandit-13", "数据库连接排障", 0.72, 1, "video", 2), ("bandit-14", "企业支付被拦截", 0.92, 0, "callback", 3), ("bandit-15", "优惠券使用规则", 0.12, 0, "article", 2), ("bandit-16", "模型API限流调试", 0.68, 1, "video", 3), ("bandit-17", "资金安全紧急咨询", 0.98, 0, "callback", 2), ("bandit-18", "配送范围查询", 0.22, 0, "article", 3)]  # 定义十八条平衡三种最佳动作的真实语义上下文。
events = []  # 保存带完整 context 和离线反事实奖励的事件。
for time, (event_id, issue, urgency, technical, ideal_arm, delay) in enumerate(event_specs):  # 按到达时刻构造 Bandit 交互。
    context = np.array([1.0, urgency, float(technical)], dtype=np.float64)  # 构造 bias、紧急度和技术标记三维上下文。
    rewards = {arm: (1.0 if arm == ideal_arm else 0.10) for arm in arms}  # 用离线反事实表给最佳动作高满意度。
    events.append({"id": event_id, "time": time, "issue": issue, "context": context, "ideal": ideal_arm, "delay": delay, "rewards": rewards})  # 保存当前交互和反馈延迟。
print("教学实验输入：客服动作延迟反馈日志")  # 标记下方为离线可复现 Bandit 数据。
print("interaction  time  issue                 context             ideal      feedback_delay")  # 输出交互预览表头。
for event in events:  # 逐条展示业务问题、上下文和反馈延迟。
    print(f"{event['id']:<12} {event['time']:>4}  {event['issue']:<22} {event['context'].tolist()!s:<19} {event['ideal']:<10} {event['delay']:>5}")  # 输出当前 Bandit 样本。
print("离线反事实奖励说明：ideal=1.0，其余=0.1；线上不能读取未选动作奖励。")  # 明确教学回放与线上可观测性边界。

教学实验输入：客服动作延迟反馈日志
interaction  time  issue                 context             ideal      feedback_delay
bandit-01       0  退款规则查询                 [1.0, 0.15, 0.0]    article        3
bandit-02       1  发票抬头查询                 [1.0, 0.2, 0.0]     article        2
bandit-03       2  SDK鉴权失败                [1.0, 0.65, 1.0]    video          4
bandit-04       3  账户被盗紧急处理               [1.0, 0.95, 0.0]    callback       2
bandit-05       4  批量接口调试                 [1.0, 0.55, 1.0]    video          3
bandit-06       5  物流时效查询                 [1.0, 0.1, 0.0]     article        2
bandit-07       6  支付接口报错                 [1.0, 0.8, 1.0]     video          3
bandit-08       7  大额扣款申诉                 [1.0, 0.9, 0.0]     callback       4
bandit-09       8  会员续费规则                 [1.0, 0.25, 0.0]    article        2
bandit-10       9  Webhook签名验证            [1.0, 0.6, 1.0]     video          3
bandit-11      10  账号冻结急需恢复               [1.0, 0.88, 0.0]    callback       2
bandit-12      11  退货地址查询 

## 2. Baseline / 基线：延迟反馈下的已观察均值贪心

未尝试 arm 的均值设为 0；三个 arm 冷启动平分时按名称顺序选择 article。article 获得任何正反馈后，贪心便不会探索 video/callback。

In [2]:
def run_delayed_greedy(events, arms):  # 模拟只用到期反馈的动作均值贪心。
    reward_sum = {arm: 0.0 for arm in arms}  # 保存每个 arm 已到期反馈总和。
    reward_count = {arm: 0 for arm in arms}  # 保存每个 arm 已到期反馈数。
    pending = []  # 保存尚未到 due_time 的 chosen 反馈。
    rows = []  # 保存逐交互选择和累计结果。
    cumulative_reward = 0.0  # 初始化实际获得总满意度。
    cumulative_regret = 0.0  # 初始化相对离线最优动作 regret。
    for event in events:  # 按时间顺序处理每个客服请求。
        due_now = [feedback for feedback in pending if feedback["due"] <= event["time"]]  # 找到当前时刻已到期反馈。
        pending = [feedback for feedback in pending if feedback["due"] > event["time"]]  # 保留未来反馈且不泄漏。
        for feedback in due_now:  # 逐条把到期奖励更新到原 chosen arm。
            reward_sum[feedback["arm"]] += feedback["reward"]  # 累加对应动作满意度。
            reward_count[feedback["arm"]] += 1  # 累加对应动作观测次数。
        means = {arm: reward_sum[arm] / reward_count[arm] if reward_count[arm] else 0.0 for arm in arms}  # 计算仅基于已到期反馈的均值。
        chosen = max(arms, key=lambda arm: (means[arm], -arms.index(arm)))  # 按均值和固定动作顺序贪心选择。
        reward = event["rewards"][chosen]  # 从离线回放表读取被选动作的实际反馈。
        oracle_reward = max(event["rewards"].values())  # 读取仅用于离线评估的最优奖励。
        cumulative_reward += reward  # 累加真实被选动作奖励。
        cumulative_regret += oracle_reward - reward  # 累加相对最佳动作的 regret。
        pending.append({"interaction_id": event["id"], "arm": chosen, "reward": reward, "due": event["time"] + event["delay"]})  # 按 interaction_id 保存延迟反馈。
        rows.append({"id": event["id"], "time": event["time"], "means": means.copy(), "chosen": chosen, "ideal": event["ideal"], "reward": reward, "applied_feedback": len(due_now), "pending": len(pending), "cumulative_reward": cumulative_reward, "cumulative_regret": cumulative_regret})  # 保存当前贪心决策和延迟状态。
    return rows, pending, reward_count  # 返回逐交互结果和最后未到期反馈。
baseline_rows, baseline_pending, baseline_counts = run_delayed_greedy(events, arms)  # 在同一十八条事件上运行贪心基线。
print("Baseline 前十条延迟贪心决策")  # 标记下表展示冷启动锁定。
print("interaction  means                                  chosen     ideal      reward  applied pending regret")  # 输出基线中间量表头。
for row in baseline_rows[:10]:  # 展示前十条选择和反馈到期过程。
    print(f"{row['id']:<12} {str({arm: round(value, 3) for arm, value in row['means'].items()}):<38} {row['chosen']:<10} {row['ideal']:<10} {row['reward']:>6.1f} {row['applied_feedback']:>7} {row['pending']:>7} {row['cumulative_regret']:>6.1f}")  # 输出当前交互的可观测均值和 regret。

Baseline 前十条延迟贪心决策
interaction  means                                  chosen     ideal      reward  applied pending regret
bandit-01    {'article': 0.0, 'video': 0.0, 'callback': 0.0} article    article       1.0       0       1    0.0
bandit-02    {'article': 0.0, 'video': 0.0, 'callback': 0.0} article    article       1.0       0       2    0.0
bandit-03    {'article': 0.0, 'video': 0.0, 'callback': 0.0} article    video         0.1       0       3    0.9
bandit-04    {'article': 1.0, 'video': 0.0, 'callback': 0.0} article    callback      0.1       2       2    1.8
bandit-05    {'article': 1.0, 'video': 0.0, 'callback': 0.0} article    video         0.1       0       3    2.7
bandit-06    {'article': 0.7, 'video': 0.0, 'callback': 0.0} article    article       1.0       1       3    2.7
bandit-07    {'article': 0.55, 'video': 0.0, 'callback': 0.0} article    video         0.1       1       3    3.6
bandit-08    {'article': 0.55, 'video': 0.0, 'callback': 0.0} article    callback   

## 3. 底层实现：每 Arm 的 A、b、theta 与 UCB Bonus

每次决策前只消费到期反馈，用原 interaction 的 context 更新 `A += xx^T`、`b += reward*x`。选择分数为 `theta^T x + alpha*sqrt(x^T A^-1 x)`。

In [3]:
def run_delayed_linucb(events, arms, alpha=1.4):  # 手写支持延迟反馈的 disjoint LinUCB。
    dimension = len(events[0]["context"])  # 读取上下文维度。
    matrices = {arm: np.eye(dimension, dtype=np.float64) for arm in arms}  # 为每个 arm 初始化正则化 A 矩阵。
    vectors = {arm: np.zeros(dimension, dtype=np.float64) for arm in arms}  # 为每个 arm 初始化 b 向量。
    pending = []  # 保存尚未到期的 interaction、context、arm 和 reward。
    rows = []  # 保存逐交互 UCB 分解和结果。
    feedback_ledger = []  # 保存每次真正应用的延迟反馈。
    cumulative_reward = 0.0  # 初始化累计满意度。
    cumulative_regret = 0.0  # 初始化累计 regret。
    for event in events:  # 按时间顺序服务十八条请求。
        due_now = [feedback for feedback in pending if feedback["due"] <= event["time"]]  # 找到当前时刻允许读取的反馈。
        pending = [feedback for feedback in pending if feedback["due"] > event["time"]]  # 保留未来反馈防止泄漏。
        for feedback in due_now:  # 逐条更新反馈所属原始 arm。
            arm = feedback["arm"]  # 读取当时实际选择的动作。
            context = feedback["context"]  # 读取当时保存的上下文而非当前上下文。
            matrices[arm] += np.outer(context, context)  # 用外积更新设计矩阵 A。
            vectors[arm] += feedback["reward"] * context  # 用奖励加权上下文更新 b。
            feedback_ledger.append({"applied_at": event["time"], "interaction_id": feedback["interaction_id"], "arm": arm, "reward": feedback["reward"], "due": feedback["due"]})  # 保存因果正确的反馈归属。
        scores = {}  # 保存三个 arm 的均值、bonus 和总 UCB。
        for arm in arms:  # 逐动作计算线性后验和置信上界。
            inverse = np.linalg.inv(matrices[arm])  # 对三乘三正定 A 求逆。
            theta = inverse @ vectors[arm]  # 计算 ridge 线性收益参数。
            mean = float(theta @ event["context"])  # 估计当前上下文期望奖励。
            bonus = float(alpha * math.sqrt(event["context"] @ inverse @ event["context"]))  # 计算未探索方向置信 bonus。
            scores[arm] = {"mean": mean, "bonus": bonus, "ucb": mean + bonus}  # 保存可解释 UCB 分项。
        chosen = max(arms, key=lambda arm: (scores[arm]["ucb"], -arms.index(arm)))  # 选择最高 UCB 并稳定处理平分。
        reward = event["rewards"][chosen]  # 离线读取被选动作满意度。
        oracle_reward = max(event["rewards"].values())  # 读取离线最优奖励供 regret 评估。
        cumulative_reward += reward  # 累加实际选择奖励。
        cumulative_regret += oracle_reward - reward  # 累加机会损失。
        pending.append({"interaction_id": event["id"], "arm": chosen, "context": event["context"].copy(), "reward": reward, "due": event["time"] + event["delay"]})  # 保存带 ID 的延迟反馈事件。
        rows.append({"id": event["id"], "time": event["time"], "scores": scores, "chosen": chosen, "ideal": event["ideal"], "reward": reward, "applied_feedback": len(due_now), "pending": len(pending), "cumulative_reward": cumulative_reward, "cumulative_regret": cumulative_regret})  # 保存当前 UCB 决策和中间状态。
    return rows, pending, feedback_ledger, matrices, vectors  # 返回轨迹、待处理反馈和最终参数。
corrected_rows, corrected_pending, feedback_ledger, final_matrices, final_vectors = run_delayed_linucb(events, arms)  # 运行延迟反馈 LinUCB。
print("LinUCB 前十条分数分解")  # 标记下表展示 mean、bonus 和 UCB。
for row in corrected_rows[:10]:  # 逐交互展示三动作分项和 pending 状态。
    compact_scores = {arm: (round(values["mean"], 3), round(values["bonus"], 3), round(values["ucb"], 3)) for arm, values in row["scores"].items()}  # 压缩三项数值供可读打印。
    print({"id": row["id"], "scores(mean,bonus,ucb)": compact_scores, "chosen": row["chosen"], "ideal": row["ideal"], "reward": row["reward"], "applied": row["applied_feedback"], "pending": row["pending"]})  # 输出当前 UCB 中间量。
print("前八条已应用反馈：", feedback_ledger[:8])  # 展示 interaction_id、选择和到期时刻的正确关联。

LinUCB 前十条分数分解
{'id': 'bandit-01', 'scores(mean,bonus,ucb)': {'article': (0.0, 1.416, 1.416), 'video': (0.0, 1.416, 1.416), 'callback': (0.0, 1.416, 1.416)}, 'chosen': 'article', 'ideal': 'article', 'reward': 1.0, 'applied': 0, 'pending': 1}
{'id': 'bandit-02', 'scores(mean,bonus,ucb)': {'article': (0.0, 1.428, 1.428), 'video': (0.0, 1.428, 1.428), 'callback': (0.0, 1.428, 1.428)}, 'chosen': 'article', 'ideal': 'article', 'reward': 1.0, 'applied': 0, 'pending': 2}
{'id': 'bandit-03', 'scores(mean,bonus,ucb)': {'article': (0.0, 2.179, 2.179), 'video': (0.0, 2.179, 2.179), 'callback': (0.0, 2.179, 2.179)}, 'chosen': 'article', 'ideal': 'video', 'reward': 0.1, 'applied': 0, 'pending': 3}
{'id': 'bandit-04', 'scores(mean,bonus,ucb)': {'article': (0.762, 1.409, 2.171), 'video': (0.0, 1.931, 1.931), 'callback': (0.0, 1.931, 1.931)}, 'chosen': 'article', 'ideal': 'callback', 'reward': 0.1, 'applied': 2, 'pending': 2}
{'id': 'bandit-05', 'scores(mean,bonus,ucb)': {'article': (0.716, 1.724, 2.4

## 4. 逐交互结果与结果解读

对相同上下文和反事实表比较贪心与 LinUCB 的动作、奖励和累计 regret。反事实奖励只服务离线教学，线上评估需 IPS/DR 等方法。

In [4]:
baseline_reward = baseline_rows[-1]["cumulative_reward"]  # 读取贪心最终累计奖励。
corrected_reward = corrected_rows[-1]["cumulative_reward"]  # 读取 LinUCB 最终累计奖励。
baseline_regret = baseline_rows[-1]["cumulative_regret"]  # 读取贪心最终累计 regret。
corrected_regret = corrected_rows[-1]["cumulative_regret"]  # 读取 LinUCB 最终累计 regret。
baseline_ideal_rate = sum(row["chosen"] == row["ideal"] for row in baseline_rows) / len(baseline_rows)  # 计算贪心最佳动作命中率。
corrected_ideal_rate = sum(row["chosen"] == row["ideal"] for row in corrected_rows) / len(corrected_rows)  # 计算 LinUCB 最佳动作命中率。
print("interaction  issue                 greedy/linucb       ideal      reward greedy/linucb  regret greedy/linucb")  # 输出逐交互同数据对照表头。
for event, baseline, corrected in zip(events, baseline_rows, corrected_rows):  # 逐条比较两种选择策略。
    print(f"{event['id']:<12} {event['issue']:<22} {baseline['chosen']:<9}/{corrected['chosen']:<9} {event['ideal']:<10} {baseline['reward']:>4.1f}/{corrected['reward']:<4.1f}          {baseline['cumulative_regret']:>4.1f}/{corrected['cumulative_regret']:<4.1f}")  # 输出当前交互的动作、收益和累计 regret。
print(f"结果解读：贪心累计reward={baseline_reward:.1f}、regret={baseline_regret:.1f}、ideal命中={baseline_ideal_rate:.1%}；LinUCB分别为{corrected_reward:.1f}、{corrected_regret:.1f}、{corrected_ideal_rate:.1%}。")  # 解释探索和上下文泛化的收益。

interaction  issue                 greedy/linucb       ideal      reward greedy/linucb  regret greedy/linucb
bandit-01    退款规则查询                 article  /article   article     1.0/1.0            0.0/0.0 
bandit-02    发票抬头查询                 article  /article   article     1.0/1.0            0.0/0.0 
bandit-03    SDK鉴权失败                article  /article   video       0.1/0.1            0.9/0.9 
bandit-04    账户被盗紧急处理               article  /article   callback    0.1/0.1            1.8/1.8 
bandit-05    批量接口调试                 article  /article   video       0.1/0.1            2.7/2.7 
bandit-06    物流时效查询                 article  /video     article     1.0/0.1            2.7/3.6 
bandit-07    支付接口报错                 article  /video     video       0.1/1.0            3.6/3.6 
bandit-08    大额扣款申诉                 article  /callback  callback    0.1/1.0            4.5/3.6 
bandit-09    会员续费规则                 article  /callback  article     1.0/0.1            4.5/4.5 
bandit-10    Webhook签名验证   

## 5. 失败案例与修正：在 due_time 之前偷看反馈

`bandit-01` 的满意度在 time=3 才到达。错误实现于 time=0 决策后立即更新 A/b，相当于读取未来；正确实现在 time=1 仍保持零个已应用反馈。

In [5]:
leak_dimension = len(events[0]["context"])  # 读取失败实验上下文维度。
leaky_matrix = np.eye(leak_dimension, dtype=np.float64)  # 初始化错误实现的 article A。
leaky_vector = np.zeros(leak_dimension, dtype=np.float64)  # 初始化错误实现的 article b。
first_context = events[0]["context"]  # 读取首个交互上下文。
first_reward = events[0]["rewards"]["article"]  # 读取首个选择的延迟满意度。
leaky_matrix += np.outer(first_context, first_context)  # 错误地在 time=0 立即使用尚未到期反馈。
leaky_vector += first_reward * first_context  # 错误地把未来奖励写入参数。
correct_matrix_at_time_one = np.eye(leak_dimension, dtype=np.float64)  # 正确实现 time=1 尚未收到反馈，A 保持先验。
correct_vector_at_time_one = np.zeros(leak_dimension, dtype=np.float64)  # 正确实现 b 仍为零。
leaked_updates = 1  # 统计错误实现提前读取的反馈条数。
correct_updates = sum(event["applied_at"] <= 1 for event in feedback_ledger)  # 从真实账本统计 time=1 已合法应用反馈数。
matrix_leak_norm = float(np.linalg.norm(leaky_matrix - correct_matrix_at_time_one))  # 量化未来信息对模型状态的污染。
print(f"错误行为：bandit-01 due={events[0]['delay']}，time=1前leaked_updates={leaked_updates}，A污染范数={matrix_leak_norm:.4f}")  # 展示延迟反馈泄漏的可测状态差。
print(f"修正行为：time=1合法updates={correct_updates}，A={correct_matrix_at_time_one.tolist()}，b={correct_vector_at_time_one.tolist()}")  # 展示 pending ledger 保持因果顺序。

错误行为：bandit-01 due=3，time=1前leaked_updates=1，A污染范数=1.0225
修正行为：time=1合法updates=0，A=[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]，b=[0.0, 0.0, 0.0]


## 6. 生产边界

离线日志知道所有动作奖励，真实线上只观察 chosen reward。生产需记录 propensity、interaction_id、模型版本和 context snapshot；处理反馈去重、超时截尾、跨天归因、IPS/DR 离线评估、安全动作约束、探索流量预算、非平稳滑窗与人工兜底。

In [6]:
bandit_diagnostics = {"interactions": len(events), "arms": len(arms), "baseline_reward": baseline_reward, "linucb_reward": corrected_reward, "baseline_regret": baseline_regret, "linucb_regret": corrected_regret, "feedback_applied": len(feedback_ledger), "feedback_pending_at_end": len(corrected_pending), "future_feedback_reads": 0}  # 汇总收益、regret 和反馈生命周期指标。
print("生产监控快照：", bandit_diagnostics)  # 输出 Contextual Bandit 服务应持续观察的信号。

生产监控快照： {'interactions': 18, 'arms': 3, 'baseline_reward': 8.099999999999998, 'linucb_reward': 11.7, 'baseline_regret': 9.900000000000002, 'linucb_regret': 6.300000000000001, 'feedback_applied': 15, 'feedback_pending_at_end': 3, 'future_feedback_reads': 0}


## 7. 最小回归测试

断言覆盖数据规模、探索多样性、累计收益、延迟归因、有限矩阵和未来信息失败。

In [7]:
assert len(events) >= 6 and len(arms) == 3  # 保证案例包含足够交互和多个动作。
assert len({row["chosen"] for row in corrected_rows}) >= 2 and len({row["chosen"] for row in baseline_rows}) == 1  # 保证 LinUCB 实际探索而贪心发生冷启动锁定。
assert corrected_reward > baseline_reward and corrected_regret < baseline_regret  # 保证同一离线回放上探索提升累计收益。
assert all(record["applied_at"] >= record["due"] for record in feedback_ledger)  # 保证每条反馈只在 due_time 后应用。
assert all(np.isfinite(matrix).all() for matrix in final_matrices.values()) and all(np.isfinite(vector).all() for vector in final_vectors.values())  # 保证 LinUCB 参数始终有限。
assert leaked_updates == 1 and correct_updates == 0 and matrix_leak_norm > 0.0  # 保证未来反馈泄漏失败真实运行且 pending 修正有效。